<a href="https://colab.research.google.com/github/mindioanni/LandLevelTools/blob/main/EGMS_shifts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Program to preprocess EGMS vertical displacement time series and build a database with detected shifts

Upload EGMS Vup csv file from your PC's directory

Install necessary packages

In [ ]:
!pip install ruptures
!pip install pyod

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyod: filename=pyod-1.1.2-py3-none-any.whl size=190289 sha256=831de08bae3e1b7402b796723546598d31638a50803753fd3ae2f698b33b45b1
  Stored in directory: /root/.cache/pip/wheels/81/1b/61/aa85b78c3c0c8871f4231e3f4a03bb23cecb7db829498380ee
Successfully built pyod


Set up the environment

In [ ]:

import ruptures as rpt
import os
import zipfile
import pandas as pd
import dask.dataframe as dd
from pyproj import Transformer
from datetime import datetime
import glob
import matplotlib.pyplot as plt
import numpy as np
from pyod.models.knn import KNN
from scipy.signal import savgol_filter
import sqlite3
import math
import time as tm


Check Environment: Add a function to determine if the code is running in Google Colab.

Set Data Directory: Automatically set the data directory based on the environment.

In [ ]:

def in_colab():
    """Check if the code is running in Google Colab."""
    try:
        import google.colab
        return True
    except ImportError:
        return False

def get_data_dir():
    """Get the data directory based on the environment."""
    if in_colab():
        print("Running in Google Colab.")
        return '/content/'  # Default directory for uploaded files in Colab
    else:
        # If not in Colab, ask the user to enter the path
        return input("Please enter the path to your data directory: ")

# Use the function to set the data directory
data_dir = get_data_dir()
print("Data directory set to:", data_dir)


Running in Google Colab.
Data directory set to: /content/


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving EGMS_L3_E53N19_100km_U_2018_2022_1_FULL.csv to EGMS_L3_E53N19_100km_U_2018_2022_1_FULL.csv


Run the basic code for shift detection.

In [ ]:
# CODE REQUIRES FURTHER CLEANING TO GET RID OFF FUNCTIONS NO LONGER NEEDED
# Program to preprocess EGMS vertical displacement time series and build a database with detected shifts

# Placeholder for the coordinate transformation subroutine
def transform_coordinates(easting, northing):
    """
    Transform coordinates from ETRS89-LAEA (EPSG:3035) to WGS84 (EPSG:4326).

    :param easting: Easting coordinate in ETRS89-LAEA
    :param northing: Northing coordinate in ETRS89-LAEA
    :return: Tuple of (longitude, latitude) in WGS84
    """
    transformer = Transformer.from_crs("EPSG:3035", "EPSG:4326", always_xy=True)

    lon, lat = transformer.transform(easting, northing)
    return round(lon, 6), round(lat, 6)

def parse_date_to_decimal_year(date_str):
    """
    Convert a date string in the format 'YYYYMMDD' to a decimal year.
    """
    # Extract year, month, and day from the date string
    year = int(date_str[:4])
    month = int(date_str[4:6])
    day = int(date_str[6:8])

    # Create datetime objects for the date and the start of that year
    date_obj = datetime(year, month, day)
    start_of_year = datetime(year, 1, 1)

    # Calculate the number of days in that year (accounting for leap years)
    if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0):  # Leap year
        days_in_year = 366
    else:
        days_in_year = 365

    # Calculate elapsed time in days since the start of the year
    elapsed_days = (date_obj - start_of_year).days

    # Convert to decimal year
    decimal_year = year + elapsed_days / days_in_year
    return round(decimal_year, 4)

# Placeholder for the PIDTS generation subroutine
def generate_pidts(row, df_columns):
    """
    Generate the PID time-series (PIDTS) from a row of the CSV file.

    :param row: A row from the CSV file containing INSAR observation data.
    :param df_columns: The column headers of the DataFrame.
    :return: A list of tuples, each containing (time in decimal years, height in mm).
    """
    time_series = []

    for i in range(11, len(row)):
        date_str = df_columns[i]  # Get the date string from the DataFrame column header
        height = row.iloc[i]  # Use iloc for integer-location based indexing

        if pd.notna(height):  # Check if height is not NaN
            decimal_year = parse_date_to_decimal_year(date_str)
            time_series.append((decimal_year, height))

    return time_series

def calculate_linear_trend_and_residuals(times, heights):
    # Linear regression
    A = np.vstack([times, np.ones(len(times))]).T
    vup, intercept = np.linalg.lstsq(A, heights, rcond=None)[0]

    # Predicted heights and residuals
    predicted_heights = vup * times + intercept
    residuals = heights - predicted_heights

    return vup, intercept, residuals, predicted_heights

def plot_time_series_and_residuals(times, heights, predicted_heights, residuals):
    plt.figure(figsize=(12, 6))

    # Plotting the time series and the linear trend
    plt.subplot(1, 2, 1)
    plt.plot(times, heights, label='Original Time Series')
    plt.plot(times, predicted_heights, color='red', label='Linear Trend')
    plt.xlabel('Time')
    plt.ylabel('Heights')
    plt.title('Time Series with Linear Trend')
    plt.legend()

    # Plotting the residuals
    plt.subplot(1, 2, 2)
    plt.scatter(times, residuals, color='orange')
    plt.axhline(y=0, color='gray', linestyle='--')
    plt.xlabel('Time')
    plt.ylabel('Residuals')
    plt.title('Residuals of Linear Trend Fit')

    plt.tight_layout()
    plt.show()

# Function to establish a database connection
def get_db_connection(db_file):
    conn = sqlite3.connect(db_file)
    return conn

def get_next_shift_number(conn, pid, t_shift):
    cursor = conn.cursor()
    cursor.execute("SELECT MAX(Shift_Number) FROM shifts WHERE PID = ? AND T_shift < ?", (pid, t_shift))
    result = cursor.fetchone()
    if result[0] is not None:
        return result[0] + 1
    return 1

def print_database_contents(db_file, table_name):
    """
    Print the contents of the specified table in the database.

    :param db_file: Path to the database file.
    :param table_name: Name of the table to query.
    """
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()

        print(f"Database contents of {table_name}:")
        for row in rows:
            print(row)
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
    finally:
        conn.close()

# Placeholder for the linear trend calculation subroutine
def calculate_linear_trend(pidts):
    """
    Calculate the linear trend and its RMS error from the PID time-series.

    :param pidts: A list of tuples, each containing (time in decimal years, height in mm).
    :return: A tuple containing (vup in mm/year, vup_rms error in mm/year).
    """
    if not pidts:
        return (0.0, 0.0)

    times, heights = zip(*pidts)
    times = np.array(times)
    heights = np.array(heights)

    # Perform linear regression
    A = np.vstack([times, np.ones(len(times))]).T
    vup, intercept = np.linalg.lstsq(A, heights, rcond=None)[0]

    # Calculate residuals (differences between observed and predicted values)
    predicted_heights = vup * times + intercept
    residuals = heights - predicted_heights

    # Calculate RMS error
    vup_rms = np.sqrt(np.mean(residuals**2))

    return (vup, vup_rms, residuals, predicted_heights)

# Placeholder for the outliers detection subroutine
def OUTLIERS(series):
    # Assuming your time-series data is stored in the 'data' array

    # Create an instance of the outlier detection model
    model = KNN(contamination=0.1)   # Specify the desired contamination level (proportion of outliers)

    # Fit the model to your time-series data
    model.fit(series)
    model_fit = model.fit(series)

    # Predict the outlier scores for each data point
    outlier_scores = model.decision_function(series)

    # Get the binary labels indicating whether each data point is an outlier or not
    outlier_labels = model.predict(series)

    return model_fit, outlier_scores, outlier_labels

# Placeholder for the filter subroutine
def APPLY_SAVITZKY_GOLAY_FILTER(time, up, window_size, poly_order):
    # Exclude NaN values from the input time-series
    valid_indices = np.where(~np.isnan(up))
    valid_time = time[valid_indices]
    valid_up = up[valid_indices]

    # Apply Savitzky-Golay filter to the valid values
    filtered_values = savgol_filter(valid_up, window_size, poly_order)

    # Create arrays of NaN values with the same shape as the input time-series
    filtered_time = np.full_like(time, np.nan)
    filtered_up = np.full_like(up, np.nan)

    # Assign the filtered values to the corresponding valid indices
    filtered_time[valid_indices] = valid_time
    filtered_up[valid_indices] = filtered_values

    return filtered_time, filtered_up

# Placeholder for the height shift detection subroutine
def OFFSET(off_up, time, method):
    """
    Detect shifts in the time-series data.
    Args:
    off_up (np.array): Array of height data.
    time (np.array): Array of time data.
    method (str): Method to use for detection ('threshold' or 'rupt').
    Returns:
    tuple: Number of shifts, times at each shift, height differences at each shift.
    """

    if method == 'thres':
        """Detect significant shifts in height in the time-series data.

        Threshold is a simple sequential change height detection algorithm.
        param pidts: A list of tuples, each containing (time in decimal years, height in mm).
        param threshold: The minimum change in height (mm) to be considered a shift.
        return: A tuple containing the number of shifts, list of times at each shift,
                 and list of height differences at each shift.
        """
        threshold = 3
        n_shifts = 0
        t_shift = []
        dh_shift = []
        offset_point = []  # Initialize an empty list for offset points

        for i in range(1, len(off_up)):
            if abs(off_up[i] - off_up[i-1]) > threshold:
                n_shifts += 1
                t_shift.append(time[i])
                # Calculate the difference and round it to 2 decimal places
                shift_difference = round(off_up[i] - off_up[i-1], 2)
                dh_shift.append(shift_difference)
                offset_point.append(i)  # Add index to offset points

        return offset_point, t_shift, dh_shift

    elif method == 'rupt':
        # Perform change point detection using the Ruptures library
        signal = off_up
        algo = rpt.Window(model="l2", width=10).fit(signal)
        result = algo.predict(pen=0.02)

        # Exclude first and last indices if present in the result
        if len(result) > 0:
            if result[0] == 0:
                result = result[1:]
            if result[-1] == len(signal) - 1:
                result = result[:-1]

        if not result:
            print("No shifts detected using rupt method.")
            return [], [], []

        # Convert result to numpy array for comparison
        offset_point = np.array(result)

        # Checking if offset points are within the length of the time array
        if offset_point[-1] >= len(time):
            print("Offset point exceeds time array bounds.")
            return [], [], []

        # Compute the best fit linear model for each segment
        segment_models = []
        for i in range(len(offset_point) - 1):
            segment_indices = np.arange(offset_point[i], offset_point[i+1])
            segment_model = np.polyfit(time[segment_indices], signal[segment_indices], 1)
            segment_models.append(segment_model)

        # Compute the differences at each offset point
        shifts = []
        for i in range(len(offset_point) - 1):
            t = time[offset_point[i]]
            model_i = segment_models[i]
            model_iplus1 = segment_models[i+1]

            y_i = model_i[0] * t + model_i[1]
            y_iplus1 = model_iplus1[0] * t + model_iplus1[1]

            # Compute the difference and round it to 2 decimal places
            shift = round(y_iplus1 - y_i, 2)
            shifts.append(shift)

        return offset_point, shifts, result

def plot_detected_shifts(times, heights, shift_points, t_shifts):
    plt.figure(figsize=(12, 6))

    # Plotting the entire time series in light blue
    plt.plot(times, heights, color='lightblue', label='Filtered Time Series')

    # Highlighting the detected shifts in red
    for shift_point in shift_points:
        plt.axvline(x=times[shift_point], color='red', linestyle='--')

    # Marking the exact time points of the shifts
    for t_shift in t_shifts:
        plt.plot(t_shift, heights[np.searchsorted(times, t_shift)], 'ro')  # 'ro' for red circle markers

    plt.xlabel('Time')
    plt.ylabel('Height')
    plt.title('Detected Shifts in Time Series')
    plt.legend()

    plt.show()

# Placeholder for the time-series correction subroutine
def correct_time_series(time_series):
    # This function corrects the time-series for height shifts
    # Returning corrected time series
    return time_series

def is_file_extracted(zip_file, data_dir):
    """
    Check if the contents of the zip file are already extracted in the data directory.
    """
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        for file_name in zip_contents:
            if not os.path.exists(os.path.join(os.path.expanduser(data_dir), file_name)):
                return False
    return True

# Function to create a table for detected shifts
def create_detected_shifts_table(conn):
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS detected_shifts (
        PID TEXT NOT NULL,
        Lon REAL NOT NULL,
        Lat REAL NOT NULL,
        Shift_Number INTEGER NOT NULL,
        T_shifts TEXT,
        DH_shifts TEXT
    )
    ''')
    conn.commit()

# Function to insert detected shifts into the detected_shifts database
def insert_detected_shifts_into_db(conn, pid, lon, lat, t_shifts, dh_shifts):
    cursor = conn.cursor()
    new_shifts_added = False  # Initialize the variable outside the if-else block

    try:
        t_shifts_str = ','.join(map(str, t_shifts))
        validated_dh_shifts = []

        for dh_shift in dh_shifts:
            try:
                validated_shift = round(float(dh_shift), 2)
                validated_dh_shifts.append(str(validated_shift))
            except ValueError:
                print(f"Invalid dh_shift value for PID {pid}: {dh_shift}")

        dh_shifts_str = ','.join(validated_dh_shifts)
        shift_number = len(t_shifts)

        cursor.execute("SELECT EXISTS(SELECT 1 FROM detected_shifts WHERE PID=?)", (pid,))
        pid_exists = cursor.fetchone()[0]

        if not pid_exists:
            cursor.execute('''
            INSERT INTO detected_shifts (PID, Lon, Lat, Shift_Number, T_shifts, DH_shifts)
            VALUES (?, ?, ?, ?, ?, ?)
            ''', (pid, lon, lat, shift_number, t_shifts_str, dh_shifts_str))
            #print(f"New PID {pid} with shifts added to detected_shifts database.")
        else:
            for t_shift, dh_shift in zip(t_shifts, validated_dh_shifts):
                cursor.execute("SELECT EXISTS(SELECT 1 FROM detected_shifts WHERE PID=? AND T_shifts LIKE ?)", (pid, f'%{t_shift}%'))
                shift_exists = cursor.fetchone()[0]

                if not shift_exists:
                    cursor.execute('''
                    UPDATE detected_shifts
                    SET Shift_Number = Shift_Number + 1, T_shifts = T_shifts || ',' || ?, DH_shifts = DH_shifts || ',' || ?
                    WHERE PID = ?
                    ''', (t_shift, dh_shift, pid))
                    new_shifts_added = True

            if new_shifts_added:
                print(f"New shifts added for existing PID {pid} in detected_shifts database.")

        conn.commit()
    except sqlite3.OperationalError as e:
        print(f"Database operational error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Function to insert shift probabilities into the shift_probabilities database
def insert_shift_probabilities_into_db(conn, pid, lon, lat, t_shifts, dh_shifts, probabilities):
    cursor = conn.cursor()
    new_probabilities_added = False  # Initialize the variable outside the if-else block

    t_shifts_str = ','.join(map(str, t_shifts))
    dh_shifts_str = ','.join(map(str, dh_shifts))
    probabilities_str = ','.join(map(str, probabilities))

    cursor.execute("SELECT EXISTS(SELECT 1 FROM shift_probabilities WHERE PID=?)", (pid,))
    pid_exists = cursor.fetchone()[0]

    if not pid_exists:
        cursor.execute('''
        INSERT INTO shift_probabilities (PID, Lon, Lat, Shift_Number, T_shifts, DH_shifts, Probabilities)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (pid, lon, lat, len(t_shifts), t_shifts_str, dh_shifts_str, probabilities_str))
        print(f"New PID {pid} with probabilities added to shift_probabilities database.")
    else:
        for t_shift, probability in zip(t_shifts, probabilities):
            cursor.execute("SELECT EXISTS(SELECT 1 FROM shift_probabilities WHERE PID=? AND T_shifts LIKE ?)", (pid, f'%{t_shift}%'))
            shift_exists = cursor.fetchone()[0]

            if not shift_exists:
                cursor.execute('''
                UPDATE shift_probabilities
                SET Shift_Number = Shift_Number + 1, T_shifts = T_shifts || ',' || ?, DH_shifts = DH_shifts || ',' || ?, Probabilities = Probabilities || ',' || ?
                WHERE PID = ?
                ''', (t_shift, dh_shifts_str, probability, pid))
                new_probabilities_added = True

        if new_probabilities_added:
            print(f"New probabilities added for existing PID {pid} in shift_probabilities database.")

    conn.commit()

# Function to read detected shifts from the detected_shifts database
def read_detected_shifts(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM detected_shifts")
    rows = cursor.fetchall()
    return rows

def main():
    # Step 0: Setup and User Input
    #data_dir = input("Please enter the path to your data directory: ")
    # Step 0: Setup for Google Colab
    data_dir = '/content/'  # Default directory in Google Colab
    overwrite_files = input("Do you want to unzip and overwrite existing files? (yes/no): ").lower()

    #locations_file = os.path.join(os.path.expanduser(data_dir), "locations.dat")
    #locations_df = pd.read_csv(locations_file)
    #print("Locations Loaded:", locations_df.head())

    # Initialize variables
    pids_with_shifts = []
    t_shifts = []
    dh_shifts = []

    # Step 1: Unzipping Files
    zip_files = glob.glob(os.path.join(os.path.expanduser(data_dir), "*.zip"))
    for zip_file in zip_files:
        if overwrite_files == 'yes':
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                print(f"Unzipping and overwriting {zip_file}...")
                zip_ref.extractall(os.path.expanduser(data_dir))
        else:
            print(f"Skipping unzipping and overwriting for {zip_file}.")

    # Initialize database connections
    conn_detected = get_db_connection(os.path.join(os.path.expanduser(data_dir), 'shifts_detected.db'))

    # Create tables in both databases
    create_detected_shifts_table(conn_detected)

    try:
        csv_files = glob.glob(os.path.join(os.path.expanduser(data_dir), "*.csv"))
        shift_method = input("Choose the shift detection method ('thres' or 'rupt'): ").lower()
        start_time = tm.time()

        for csv_file in csv_files:
            df = pd.read_csv(csv_file)
            #print(f"Processing {csv_file} with {len(df['pid'].unique())} unique PIDs")

            for _, row in df.iterrows():
                pid = row['pid']
                # Add the PID to the list of processed PIDs
                if pid not in pids_with_shifts:
                    pids_with_shifts.append(pid)
                easting = row['easting']
                northing = row['northing']

                # Step 2: Transform coordinates and generate PID time-series
                lon, lat = transform_coordinates(easting, northing)
                pidts = generate_pidts(row, df.columns)

                # Step 3: Detect and remove outliers
                times, heights = zip(*pidts)
                heights_array = np.array(heights)
                time = np.array(times)
                series = np.column_stack((time, heights_array))
                _, _, result_outlier_labels = OUTLIERS(series)
                cleaned_heights = heights_array[result_outlier_labels == 0]
                cleaned_time = time[result_outlier_labels == 0]

                # Step 4: Filter the cleaned time series
                filtered_time, filtered_heights = APPLY_SAVITZKY_GOLAY_FILTER(cleaned_time, cleaned_heights, window_size=3, poly_order=1)

                # Calculate the linear trend and residuals
                vup, intercept, residuals, predicted_heights = calculate_linear_trend_and_residuals(filtered_time, filtered_heights)

                # Plot the time series with the linear trend and the residuals
                #plot_time_series_and_residuals(filtered_time, filtered_heights, predicted_heights, residuals)

                try:
                    # Step 5: Detect offsets in the filtered time series
                    if shift_method == 'rupt':
                        offset_point, shifts_detected, dh_shift_detected = OFFSET(filtered_heights, filtered_time, method='rupt')
                    else:
                        offset_point, shifts_detected, dh_shift_detected = OFFSET(filtered_heights, filtered_time, method='thres')

                    # Ensure offset points are within bounds
                    offset_point = [pt for pt in offset_point if pt < len(filtered_time)]

                    # Redefine t_shift and dh_shift based on offset_point
                    if len(offset_point) > 0:
                        t_shift = [filtered_time[pt] for pt in offset_point]
                        dh_shift = [dh_shift_detected[i] for i in range(len(offset_point))]
                        # FOR DEBUG USE ONLY Verify the values
                        #print(f"After OFFSET function, t_shifts: {t_shift}, dh_shifts: {dh_shift}")
                    else:
                        # Handle case where no shifts are detected - skip to next PID
                        #print(f"No shifts detected for PID {pid}. Moving to next PID.")
                        continue  # Skip the rest of the loop and proceed to the next iteration

                    # Plot the time series with detected shifts
                    #plot_detected_shifts(filtered_time, filtered_heights, offset_point, t_shift)

                    # Write offset-related data to the database and store values for later probability calculation
                    if len(offset_point) > 0:
                        t_shift = [filtered_time[pt] for pt in offset_point]
                        dh_shift = [dh_shift_detected[i] for i in range(len(offset_point))]
                        #print(f"Inserting detected shifts for PID {pid} into shifts_detected.db")
                        insert_detected_shifts_into_db(conn_detected, pid, lon, lat, t_shift, dh_shift)


                except Exception as e:
                    print(f"An error occurred while processing row: {e}")
                    # Optional: Add break here to stop processing further after an error


    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        # Close the database connections after all files have been processed
        conn_detected.close()

    # Specify the database files for printing and plotting
    #detected_db_file = os.path.join(os.path.expanduser(data_dir), 'shifts_detected.db')

    # Print for detected_shifts database
    #print("Contents of Detected Shifts Database:")
    #print_database_contents(detected_db_file, "detected_shifts")

    end_time = tm.time()  # Stop the timer
    elapsed_time = end_time - start_time  # Calculate elapsed time

    num_pids_processed = len(pids_with_shifts)  # Count the number of PIDs processed
    print(f"Program completed in {elapsed_time:.2f} seconds. Number of PIDs processed: {num_pids_processed}")

if __name__ == "__main__":
    main()


Do you want to unzip and overwrite existing files? (yes/no): no
Choose the shift detection method ('thres' or 'rupt'): thres


Now lets run the program for building up the Base to Radial PID's distances database

In [ ]:

# Haversine formula to calculate distance between two points on the earth
def haversine(lon1, lat1, lon2, lat2):
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    # DEBUG Print the contents
    #print(a)
    #print(c)
    r = 6371  # Radius of Earth in kilometers
    return round(c * r, 3)

def create_pid_distances_table(conn):
    """
    Creates a table in the database to store distances between PIDs.
    """
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS pid_distances (
        PID1 TEXT NOT NULL,
        PID2 TEXT NOT NULL,
        Distance REAL NOT NULL
    )
    ''')
    conn.commit()

def insert_distance(conn, pid1, pid2, distance):
    """
    Inserts a distance record into the database if it doesn't already exist.
    """
    cursor = conn.cursor()

    # Check if the distance record already exists
    cursor.execute("SELECT * FROM pid_distances WHERE PID1=? AND PID2=?", (pid1, pid2))
    if not cursor.fetchone():
        # If record does not exist, insert new record
        cursor.execute("INSERT INTO pid_distances (PID1, PID2, Distance) VALUES (?, ?, ?)", (pid1, pid2, distance))
        conn.commit()

def build_distances_database(source_db_path, target_db_path):
    pid_count = 0
    distances_count = 0

    try:
        source_conn = sqlite3.connect(source_db_path)
        target_conn = sqlite3.connect(target_db_path)

        create_pid_distances_table(target_conn)

        source_cursor = source_conn.cursor()
        source_cursor.execute("SELECT PID, Lon, Lat FROM detected_shifts")
        all_pids = source_cursor.fetchall()
        pid_count = len(all_pids)

        for i, (pid1, lon1, lat1) in enumerate(all_pids):
            for j, (pid2, lon2, lat2) in enumerate(all_pids):
                if j > i:  # To avoid duplicate calculations and self-distance
                    distance = haversine(lon1, lat1, lon2, lat2)
                    if distance <= 1.0:  # set a maximum distance
                        insert_distance(target_conn, pid1, pid2, distance)
                        insert_distance(target_conn, pid2, pid1, distance)  # Reverse direction
                        distances_count += 2

        source_conn.close()
        target_conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")

    return pid_count, distances_count

def print_distances_database_contents(db_file):
    try:
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()

        cursor.execute("SELECT * FROM pid_distances")
        rows = cursor.fetchall()

        print("Contents of the PID Distances Database:")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
    finally:
        conn.close()

def main():
    start_time = tm.time()

    # User input for the data directory
    # Step 0: Setup for Google Colab
    #data_dir = input("Please enter the path to your data directory: ")
    data_dir = '/content/'  # Default directory in Google Colab

    # Paths for the source and target databases
    source_db = os.path.join(os.path.expanduser(data_dir), 'shifts_detected.db')
    target_db = os.path.join(os.path.expanduser(data_dir), 'pid_distances.db')

    # Build distances database
    pid_count, distances_count = build_distances_database(source_db, target_db)
    elapsed_time = tm.time() - start_time

    # Print execution details
    print(f"Database built in {elapsed_time:.2f} seconds.")
    print(f"Number of PIDs processed: {pid_count}")
    print(f"Number of distances computed: {distances_count}")

    # DEBUG Print the contents of the distances database
    #print_distances_database_contents(target_db)

if __name__ == "__main__":
    main()


Database built in 0.00 seconds.
Number of PIDs processed: 12
Number of distances computed: 132


Program to compute the probabilities based on Influence Factor Algorithm, and store the results to shift_propbabilities.db database

In [ ]:
# Initializing functions:

# Create database for storing shift probabilities
def create_probabilities_table(conn):
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS shifts_probabilities (
        PID TEXT NOT NULL,
        Lon REAL NOT NULL,
        Lat REAL NOT NULL,
        Shift_Number INTEGER NOT NULL,
        Number_Radials INTEGER NOT NULL,
        Common_Incidents INTEGER NOT NULL,
        T_shift REAL,
        DH_shift REAL,
        Probability REAL
    )
    ''')
    conn.commit()

# Establish connection to the databases in the system holding detected shifts and distances
def get_base_pid_data(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM detected_shifts LIMIT 1")
    return cursor.fetchone()

def get_shift_data(conn, pid):
    cursor = conn.cursor()
    cursor.execute("SELECT T_shifts, DH_shifts FROM detected_shifts WHERE PID = ?", (pid,))
    result = cursor.fetchone()
    if result:
        t_shifts = [float(t) for t in result[0].split(',')]
        dh_shifts = [float(dh) for dh in result[1].split(',')]
        return t_shifts, dh_shifts
    return [], []

def get_radial_pids(conn, base_pid, radius):
    cursor = conn.cursor()
    cursor.execute("SELECT PID2, Distance FROM pid_distances WHERE PID1 = ? AND Distance < ?", (base_pid, radius))
    return cursor.fetchall()

def print_shifts_probabilities_contents(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM shifts_probabilities")
    records = cursor.fetchall()
    if records:
        print("Contents of shifts_probabilities database:")
        for record in records:
            print(record)
    else:
        print("No data in shifts_probabilities database.")

def main():
    SEARCH_RADIUS = 0.4  # in km
    #db_path = input("Enter the system path to your data: ")
    # Step 0: Setup for Google Colab
    db_path = '/content/'  # Default directory in Google Colab
    db_path = os.path.expanduser(db_path)

    start_time = tm.time()

    if not (os.path.exists(os.path.join(db_path, "shifts_detected.db")) and os.path.exists(os.path.join(db_path, "pid_distances.db"))):
        print("Error: Required database files not found in the provided path.")
        return

    conn_shifts_probabilities = sqlite3.connect(os.path.join(db_path, 'shifts_probabilities.db'))
    conn_shifts_detected = sqlite3.connect(os.path.join(db_path, 'shifts_detected.db'))
    create_probabilities_table(conn_shifts_probabilities)
    base_pid_data = get_base_pid_data(conn_shifts_detected)

    if not base_pid_data:
        print("No data found in detected_shifts database.")
        return

    radius = 0.4
    conn_pid_distances = sqlite3.connect(os.path.join(db_path, 'pid_distances.db'))
    cursor_shifts_detected = conn_shifts_detected.cursor()
    cursor_shifts_detected.execute("SELECT PID FROM detected_shifts")
    all_pids = cursor_shifts_detected.fetchall()

    for pid_row in all_pids:
        base_pid = pid_row[0]
        #print(f"Base PID is: {base_pid}")

        radial_pids = get_radial_pids(conn_pid_distances, base_pid, radius)
        num_radials = len(radial_pids)

        if radial_pids:
            #print(f"Radial PIDs for {base_pid}: {radial_pids}")
            min_distance = min(radial_pids, key=lambda x: x[1])[1]
            #print(f"Minimum distance for {base_pid}: {min_distance}")

            base_t_shifts, base_dh_shifts = get_shift_data(conn_shifts_detected, base_pid)
            num_shifts = len(base_t_shifts)

            for t_shift in base_t_shifts:
                shift_differences = []
                for radial_pid, distance in radial_pids:
                    radial_t_shifts, radial_dh_shifts = get_shift_data(conn_shifts_detected, radial_pid)
                    if t_shift in radial_t_shifts:
                        radial_index = radial_t_shifts.index(t_shift)
                        radial_dh_shift = radial_dh_shifts[radial_index]
                        shift_difference = abs(base_dh_shifts[base_t_shifts.index(t_shift)] - radial_dh_shift)
                        shift_differences.append(shift_difference)

                if shift_differences:
                    mean_shift_dif = sum(shift_differences) / len(shift_differences)
                    influence_scores = []

                    for radial_pid, distance in radial_pids:
                        if t_shift in get_shift_data(conn_shifts_detected, radial_pid)[0]:
                            influence_score = (min_distance / distance) * 1 / (abs(shift_difference - mean_shift_dif) + 1)
                            influence_scores.append(influence_score)

                    probability = sum(influence_scores) / len(radial_pids)
                    #print(f"Probability, after {len(radial_pids)} Radial PIDs and {len(influence_scores)} common incidents, for shift time {t_shift}: {probability}")

                    # Store the results in shifts_probabilities.db
                    cursor = conn_shifts_probabilities.cursor()
                    cursor.execute("INSERT INTO shifts_probabilities (PID, Lon, Lat, Shift_Number, Number_Radials, Common_Incidents, T_shift, DH_shift, Probability) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                                   (base_pid, base_pid_data[1], base_pid_data[2], num_shifts, num_radials, len(shift_differences), t_shift, base_dh_shifts[base_t_shifts.index(t_shift)], probability))
                    conn_shifts_probabilities.commit()
        else:
            #print("No radial PIDS found for this base PID.")
            cursor = conn_shifts_probabilities.cursor()
            for t_shift, dh_shift in zip(base_t_shifts, base_dh_shifts):
                cursor.execute("INSERT INTO shifts_probabilities (PID, Lon, Lat, Shift_Number, Number_Radials, Common_Incidents, T_shift, DH_shift, Probability) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                               (base_pid, base_pid_data[1], base_pid_data[2], num_shifts, num_radials, 0, t_shift, dh_shift, 0.0))
            conn_shifts_probabilities.commit()

    conn_shifts_detected.close()
    conn_pid_distances.close()
    print_shifts_probabilities_contents(conn_shifts_probabilities)
    conn_shifts_probabilities.close()
    #print("Database 'shifts_probabilities.db' has been set up successfully.")

    elapsed_time = tm.time() - start_time
    # Print execution details
    print(f"Database built in {elapsed_time:.2f} seconds.")

if __name__ == "__main__":
    main()


Contents of shifts_probabilities database:
('10NemiOAb2', 22.198749, 39.794124, 3, 9, 9, 2021.1671, 8.93, 0.31444504945388707)
('10NemiOAb2', 22.198749, 39.794124, 3, 9, 3, 2021.1836, 9.43, 0.1048992788088735)
('10NemiOAb2', 22.198749, 39.794124, 3, 9, 3, 2021.2329, 8.77, 0.14216838185231517)
('10NerP3Pn6', 22.198749, 39.794124, 3, 9, 9, 2021.1671, 8.87, 0.4003745474476602)
('10NerP3Pn6', 22.198749, 39.794124, 3, 9, 4, 2021.2, 9.53, 0.11432916992227231)
('10NerP3Pn6', 22.198749, 39.794124, 3, 9, 3, 2021.2164, 8.87, 0.1634522537532569)
('10NerP3Pn5', 22.198749, 39.794124, 3, 8, 8, 2021.1671, 9.03, 0.42251658155759864)
('10NerP3Pn5', 22.198749, 39.794124, 3, 8, 1, 2021.2493, 9.1, 0.05656108597285068)
('10NerP3Pn4', 22.198749, 39.794124, 3, 5, 5, 2021.1671, 9.53, 0.3988074589263706)
('10NerP3Pn4', 22.198749, 39.794124, 3, 5, 1, 2021.1836, 8.97, 0.08888888888888889)
('10NerP3Pn4', 22.198749, 39.794124, 3, 5, 1, 2021.2, 8.7, 0.10050251256281406)
('10New5iezA', 22.198749, 39.794124, 3, 10, 1

If you need to upload any data from the contents to your Google Drive...

Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil

def copy_to_drive(source_path, destination_path):
    """
    Copy a file from the Colab environment to Google Drive.

    Parameters:
    source_path (str): Path to the source file in the Colab environment.
    destination_path (str): Path to the destination directory in Google Drive.
    """
    # Create the destination path in Google Drive if it doesn't exist
    destination_dir = os.path.dirname(destination_path)
    if not os.path.exists(destination_dir):
        os.makedirs(destination_dir)

    # Copy the file
    shutil.copy(source_path, destination_path)


REMOVE DATA FROM CONTENTS IF YOU WANT TO

In [ ]:

def remove_egms_files():
    # List of specific database files
    db_files = ['shifts_detected.db', 'pid_distances.db', 'shifts_probabilities.db']

    # Find all files with "*EGMS*" in the filename and a ".csv" extension
    egms_files = glob.glob('*EGMS*.csv')

    # Combine database files and EGMS files into a single list
    files_to_remove = db_files + egms_files

    # Prompt the user
    print("The following files are identified for removal:")
    for file in files_to_remove:
        print(file)

    confirm = input("Do you want to remove EGMS Vup observations file and the generated database files from your system? (yes/no): ").lower()

    if confirm == 'yes':
        for file in files_to_remove:
            if os.path.exists(file):
                os.remove(file)
                print(f"Removed {file}")
            else:
                print(f"File {file} not found.")
        print("All specified files were removed successfully.")
    else:
        print("File removal cancelled.")

# Example usage
remove_egms_files()


The following files are identified for removal:
shifts_detected.db
pid_distances.db
shifts_probabilities.db
Do you want to remove EGMS Vup observations file and the generated database files from your system? (yes/no): yes
File shifts_detected.db not found.
File pid_distances.db not found.
Removed shifts_probabilities.db
All specified files were removed successfully.
